# Stage 5 - adversarial training (defence)

Adversarially train the CNN (augment clean + adversarial) and compare the defended model against the baseline CNN and the Random Forest under PGD. This is where the datasets diverge: AT helps the scarce CICIoV classes and does not help the diverse ROAD ones.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Setup

In [ ]:
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn, build_random_forest
from adversec.experiments import attack as atk
from adversec.experiments.defense import adversarial_train_cnn
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
def mf1(y, p, labels=None): return f1_score(y, p, labels=labels, average='macro', zero_division=0)

## Train baseline CNN, defended CNN (PGD-augmented) and RF, then attack at PGD eps=0.10
For CICIoV the headline is robust-support macro-F1 (classes with >=2 test frames); for ROAD it is the plain macro-F1.

In [ ]:
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    Xtr, ytr, Xte, yte = a['X_train'], a['y_train'], a['X_test'].astype(np.float32), a['y_test']
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    print(f'\n########## {name} ##########')
    base = train_cnn(CNN1D(n_features=Xtr.shape[1], n_classes=len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    print('-- adversarial training (PGD-augmented) --')
    defended = adversarial_train_cnn(Xtr, ytr, strategy='pgd', n_epochs=50, device=DEVICE, class_weights=cw)
    rf = build_random_forest().fit(Xtr, ytr)
    base_clf = atk.wrap_cnn_for_art(base, n_features=Xtr.shape[1], n_classes=len(classes), device=DEVICE)
    def_clf  = atk.wrap_cnn_for_art(defended, n_features=Xtr.shape[1], n_classes=len(classes), device=DEVICE)
    Xadv = atk.generate_pgd(base_clf, Xte, 0.10)   # test-time attack from the baseline
    rs = cfg.get('defence', {}).get('robust_support')
    labels = [classes.index(c) for c in rs] if rs else None
    metric = 'robust-support macro-F1' if labels else 'macro-F1'
    def score(clf, X): return mf1(yte, clf.predict(X).argmax(1), labels)
    print(f'\n{name}: {metric}   (attack = PGD eps=0.10, crafted from baseline)')
    print(f"  {'model':14s}{'clean':>9}{'attacked':>10}")
    for lab, clf in [('baseline CNN', base_clf), ('defended CNN', def_clf)]:
        print(f'  {lab:14s}{score(clf, Xte):>9.3f}{score(clf, Xadv):>10.3f}')
    print(f"  {'Random Forest':14s}{mf1(yte, rf.predict(Xte), labels):>9.3f}{mf1(yte, rf.predict(Xadv), labels):>10.3f}")